In [1]:
import os
import urllib.request
import zipfile

# Base URL for raw files from the official NGCF repo
base_url = "https://raw.githubusercontent.com/xiangwang1223/neural_graph_collaborative_filtering/master/Data"

# All three datasets
datasets = {
    "gowalla": ["train.txt", "test.txt"],
    "yelp2018": ["train.txt", "test.txt"],
    "amazon-book": ["train.txt", "test.txt"],
}

project_root = "/Users/joshua/Desktop/ngcf"  # adjust if different
data_root = os.path.join(project_root, "data")

for dataset, files in datasets.items():
    folder = os.path.join(data_root, dataset)
    os.makedirs(folder, exist_ok=True)
    
    for fname in files:
        url = f"{base_url}/{dataset}/{fname}"
        dest = os.path.join(folder, fname)
        
        if os.path.exists(dest):
            print(f"  Already exists: {dest}")
            continue
            
        print(f"Downloading {dataset}/{fname} ...")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"  ✓ Saved to {dest}")
        except Exception as e:
            print(f"  ✗ FAILED: {e}")

print("\nDone!")

  ✓ Saved to /Users/joshua/Desktop/ngcf/data/gowalla/train.txt
  ✓ Saved to /Users/joshua/Desktop/ngcf/data/gowalla/test.txt
  ✗ FAILED: HTTP Error 404: Not Found
  ✗ FAILED: HTTP Error 404: Not Found
  ✓ Saved to /Users/joshua/Desktop/ngcf/data/amazon-book/train.txt
  ✓ Saved to /Users/joshua/Desktop/ngcf/data/amazon-book/test.txt

Done!


In [2]:
# Verify all files exist and print stats
for dataset in ["gowalla", "yelp2018", "amazon-book"]:
    train_path = os.path.join(data_root, dataset, "train.txt")
    test_path = os.path.join(data_root, dataset, "test.txt")
    
    print(f"\n{'='*50}")
    print(f"  {dataset.upper()}")
    print(f"{'='*50}")
    
    for label, path in [("train", train_path), ("test", test_path)]:
        if not os.path.exists(path):
            print(f"  ✗ {label}.txt MISSING!")
            continue
        
        users = set()
        items = set()
        n_interactions = 0
        
        with open(path, 'r') as f:
            for line in f:
                tokens = line.strip().split()
                if len(tokens) < 2:
                    continue
                user = int(tokens[0])
                user_items = [int(t) for t in tokens[1:]]
                users.add(user)
                items.update(user_items)
                n_interactions += len(user_items)
        
        print(f"  {label}.txt: {len(users):,} users | {len(items):,} items | {n_interactions:,} interactions")
    
    # Show first line as sanity check
    with open(train_path, 'r') as f:
        first = f.readline().strip()
    print(f"  First line: {first[:80]}...")


  GOWALLA
  train.txt: 29,858 users | 40,981 items | 810,128 interactions
  test.txt: 29,858 users | 38,546 items | 217,242 interactions
  First line: 0 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 2...

  YELP2018
  ✗ train.txt MISSING!
  ✗ test.txt MISSING!


FileNotFoundError: [Errno 2] No such file or directory: '/Users/joshua/Desktop/ngcf/data/yelp2018/train.txt'

In [5]:
!cd ~/Desktop/ngcf && python config.py




python: can't open file '/Users/joshua/Desktop/ngcf/config.py': [Errno 2] No such file or directory


In [6]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python config.py

Namespace(dataset='gowalla', data_path='./data/', embed_size=64, n_layers=3, layer_sizes=[64, 64, 64], lr=0.0001, batch_size=1024, epoch=400, reg=1e-05, mess_dropout=0.1, node_dropout=0.0, Ks=[20], patience=50, seed=2019, eval_interval=5, device='auto')


In [8]:
import sys
sys.path.insert(0, '/Users/joshua/Desktop/REC_SYS_PROJECT')

from utils.data_loader import NGCFDataset, get_dataloader

# Load Gowalla
dataset = NGCFDataset('./data/', 'gowalla')

# Test the dataloader
loader = get_dataloader(dataset, batch_size=1024, num_workers=0)
batch = next(iter(loader))
users, pos_items, neg_items = batch

print(f"\nBatch shapes: users={users.shape}, pos={pos_items.shape}, neg={neg_items.shape}")
print(f"Sample triple: user={users[0].item()}, pos={pos_items[0].item()}, neg={neg_items[0].item()}")
print(f"Interaction matrix shape: {dataset.interaction_matrix.shape}")
print(f"Interaction matrix nnz: {dataset.interaction_matrix.nnz:,}")



  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

Batch shapes: users=torch.Size([1024]), pos=torch.Size([1024]), neg=torch.Size([1024])
Sample triple: user=29483, pos=38666, neg=16618
Interaction matrix shape: (29858, 40981)
Interaction matrix nnz: 810,128


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [9]:
from utils.adjacency import build_normalized_laplacian

L = build_normalized_laplacian(dataset.interaction_matrix, dataset.n_users, dataset.n_items)

print(f"Type: {type(L)}")
print(f"Shape: {L.shape}")
print(f"Expected shape: ({dataset.n_users + dataset.n_items}, {dataset.n_users + dataset.n_items})")
print(f"Non-zeros: {L._nnz():,}")
print(f"Expected nnz: {dataset.n_train * 2:,}  (2 x n_train because symmetric)")
print(f"Sample values (should be around 0.001-0.1): {L.values()[:5]}")


  Laplacian: shape=(70839, 70839), nnz=1,620,256
Type: <class 'torch.Tensor'>
Shape: torch.Size([70839, 70839])
Expected shape: (70839, 70839)
Non-zeros: 1,620,256
Expected nnz: 1,620,256  (2 x n_train because symmetric)
Sample values (should be around 0.001-0.1): tensor([0.0246, 0.0246, 0.0209, 0.0314, 0.0296])


In [10]:
from models.ngcf import NGCF

model = NGCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    embed_size=64,
    layer_sizes=[64, 64, 64],
    laplacian=L,
    node_dropout=0.0,
    mess_dropout=0.1
)

print(f"Final embed size: {model.get_final_embed_size()}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")
print(f"Embedding parameters: {model.embedding.weight.numel():,}")
print(f"Weight parameters: {n_params - model.embedding.weight.numel():,}")

# Test forward pass with one batch
model.train()
u_emb, p_emb, n_emb, u_ego, p_ego, n_ego = model(users, pos_items, neg_items)
print(f"\nForward pass output shapes:")
print(f"  u_emb:  {u_emb.shape}   (expected: [1024, 256])")
print(f"  p_emb:  {p_emb.shape}   (expected: [1024, 256])")
print(f"  u_ego:  {u_ego.shape}   (expected: [1024, 64])")

# Test eval mode
model.eval()
user_emb, item_emb = model.get_all_embeddings()
print(f"\nEval embeddings:")
print(f"  user_emb: {user_emb.shape}  (expected: [{dataset.n_users}, 256])")
print(f"  item_emb: {item_emb.shape}  (expected: [{dataset.n_items}, 256])")


Final embed size: 256
Total parameters: 4,558,656
Embedding parameters: 4,533,696
Weight parameters: 24,960

Forward pass output shapes:
  u_emb:  torch.Size([1024, 256])   (expected: [1024, 256])
  p_emb:  torch.Size([1024, 256])   (expected: [1024, 256])
  u_ego:  torch.Size([1024, 64])   (expected: [1024, 64])

Eval embeddings:
  user_emb: torch.Size([29858, 256])  (expected: [29858, 256])
  item_emb: torch.Size([40981, 256])  (expected: [40981, 256])


In [12]:
import torch
from utils.metrics import evaluate_model

model.eval()
res = evaluate_model(model, dataset, [20], device=torch.device('cpu'), batch_size=256)
print("Random init evaluation:")
for k, v in res.items():
    print(f"  {k}: {v:.4f}")

Random init evaluation:
  recall@20: 0.0008
  ndcg@20: 0.0007


In [13]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 3 --eval_interval 3 --device cpu

Device: cpu
Config: {'dataset': 'gowalla', 'data_path': './data/', 'embed_size': 64, 'n_layers': 3, 'layer_sizes': [64, 64, 64], 'lr': 0.0001, 'batch_size': 1024, 'epoch': 3, 'reg': 1e-05, 'mess_dropout': 0.1, 'node_dropout': 0.0, 'Ks': [20], 'patience': 50, 'seed': 2019, 'eval_interval': 3, 'save_path': './outputs/', 'verbose': 1, 'device': 'cpu'}

[1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[2/5] Building normalized Laplacian ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[3/5] Initializing NGCF model ...
  Parameters: 4,558,656
  Final embed dim: 256

[4/5] Training (max 3 epochs, patience 50) ...

/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
^C
Traceback 

In [1]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 3 --eval_interval 3 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       1024
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 3 epochs) ...
----------------------------------------------------------------------
 Epoch |       

In [9]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 400 --eval_interval 10 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       1024
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |     

In [10]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 3 --eval_interval 3 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       1024
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 3 epochs) ...
----------------------------------------------------------------------
 Epoch |       

In [11]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 3 --eval_interval 3 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 3 epochs) ...
----------------------------------------------------------------------
 Epoch |       

In [12]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 400 --eval_interval 10 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |     

In [13]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 400 --eval_interval 10 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |     

In [14]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset yelp2018 --epoch 400 --eval_interval 10 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          yelp2018
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      31,668
  #Items:      38,048
  #Train:   1,237,259
  #Test:      324,147
  Density:    0.00103

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(69716, 69716), nnz=2,474,518

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,486,784
  Embedding parameters: 4,461,824
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |    

In [15]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset amazon-book --epoch 400 --eval_interval 10 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...
  Total parameters:     9,256,448
  Embedding parameters: 9,231,488
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch 

In [16]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset amazon-book --epoch 400 --eval_interval 10 --batch_size 8192 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       8192
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...
  Total parameters:     9,256,448
  Embedding parameters: 9,231,488
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch 

In [17]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python pretrain_mf.py --dataset gowalla --batch_size 4096

  MF Pre-training for NGCF
  Dataset: gowalla
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

Training MF for 200 epochs...
--------------------------------------------------
  Epoch   10 | loss 0.10244 | 2.7s
  Epoch   20 | loss 0.05579 | 2.7s
  Epoch   30 | loss 0.04157 | 2.7s
  Epoch   40 | loss 0.03349 | 2.8s
  Epoch   50 | loss 0.02786 | 2.8s
  Epoch   60 | loss 0.02392 | 2.8s
  Epoch   70 | loss 0.02126 | 2.8s
  Epoch   80 | loss 0.01864 | 2.8s
  Epoch   90 | loss 0.01652 | 2.8s
  Epoch  100 | loss 0.01487 | 2.8s
  Epoch  110 | loss 0.01355 | 2.8s
  Epoch  120 | loss 0.01210 | 2.8s
  Epoch  130 | loss 0.01116 | 2.8s
  Epoch  140 | loss 0.01051 | 2.8s
  Epoch  150 | loss 0.00952 | 2.8s
  Epoch  160 | loss 0.00896 | 2.8s
  Epoch  170 | loss 0.00830 | 2.8s
  Epoch  180 | loss 0.00788 | 2.8s
  Epoch  190 | loss 0.00751 | 2.8s
  Epoch  200 | loss 0.00706 | 2.8s
---------------------------------------------

##### !cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 400 --eval_interval 10 --batch_size 4096 --pretrain ./outputs/mf_gowalla_pretrain.pth --device cpu

In [19]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --epoch 400 --eval_interval 10 --batch_size 4096 --pretrain ./outputs/mf_gowalla_pretrain.pth --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           3 x [64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Pre-trained embeddings: ./outputs/mf_gowalla_pretrain.pth
  >> Loaded pre-trained MF embeddings
  Total parameters:     4,558,656
  Embedding parameters: 4,533,696
  Weight parameters:    24,960
  Final embed dim:      256

[Step 4/5] Training (max 40

In [20]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python generate_all_results.py

  NGCF PAPER REPRODUCTION — ALL RESULTS AND FIGURES
  Generating everything from trained models and logs...

  TABLE 1: Dataset Statistics
Dataset             #Users     #Items   #Interactions      Density
----------------------------------------------------------------------
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066
gowalla             29,858     40,981       1,027,370      0.00084
  Dataset loaded
  #Users:      31,668
  #Items:      38,048
  #Train:   1,237,259
  #Test:      324,147
  Density:    0.00103
yelp2018            31,668     38,048       1,561,406      0.00130
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049
amazon-book         52,643     91,599       2,984,108      0.00062

Paper reference (Table 1):
Gowalla             29,858     40,981       1,027,370      0.00084
Yelp2018*           31,668     38,048       1,

In [22]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python -c "
import os, numpy as np, torch, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from utils.data_loader import NGCFDataset

dataset = NGCFDataset('./data/', 'gowalla')

# Load MF pretrained embeddings (= NGCF-0 = MF baseline)
mf_emb = torch.load('./outputs/mf_gowalla_pretrain.pth', map_location='cpu', weights_only=True)
user_emb = mf_emb[:dataset.n_users]
item_emb = mf_emb[dataset.n_users:]

np.random.seed(42)
candidates = [u for u in dataset.test_data.keys() if len(dataset.test_data[u]) >= 5]
selected = np.random.choice(candidates, size=6, replace=False)

points, labels, types = [], [], []
for uid in selected:
    points.append(user_emb[uid].numpy())
    labels.append(uid)
    types.append('user')
    for iid in dataset.test_data.get(uid, [])[:10]:
        if iid < item_emb.shape[0]:
            points.append(item_emb[iid].numpy())
            labels.append(uid)
            types.append('item')

points = np.array(points)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(points)-1))
pts2d = tsne.fit_transform(points)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
fig, ax = plt.subplots(figsize=(8, 8))
for i, (pt, lbl, tp) in enumerate(zip(pts2d, labels, types)):
    cidx = list(selected).index(lbl)
    c = colors[cidx]
    if tp == 'user':
        ax.scatter(pt[0], pt[1], c=[c], marker='*', s=200, edgecolors='black', linewidths=1, zorder=5)
    else:
        ax.scatter(pt[0], pt[1], c=[c], marker='o', s=40, alpha=0.7)

legend_els = [plt.scatter([], [], c=[colors[i]], marker='*', s=100, label=f'User {u}') for i, u in enumerate(selected)]
ax.legend(handles=legend_els, loc='upper right', fontsize=9)
ax.set_title('(a) MF (NGCF-0) — t-SNE Embedding Visualization')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.grid(True, alpha=0.2)
plt.savefig('./outputs/figures/figure7_tsne_mf.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: ./outputs/figures/figure7_tsne_mf.png')

zsh:1: unmatched "
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066
Saved: ./outputs/figures/figure7_tsne_mf.png


In [1]:
!cd /Users/joshua/Desktop/REC_SYS_PROJECT && python main.py --dataset gowalla --n_layers 1 --layer_sizes 64 --epoch 400 --eval_interval 10 --batch_size 4096 --device cpu

  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           1 x [64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Message dropout:  0.1
  Node dropout:     0.0
  Eval metric:      recall@20, ndcg@20
  Early stopping:   50 epochs patience
  Seed:             2019

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5] Initializing NGCF model ...
  Total parameters:     4,542,016
  Embedding parameters: 4,533,696
  Weight parameters:    8,320
  Final embed dim:      128

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss   

In [ ]:
import subprocess
import os
import time

# 1. Configuration
project_dir = "/Users/joshua/Desktop/REC_SYS_PROJECT"
os.chdir(project_dir)

# Define what needs to be run for Table 3
# Gowalla L=1 is already done. L=3 for all is already done.
tasks = [
    {"dataset": "gowalla", "layers": [2, 4]},
    {"dataset": "yelp2018", "layers": [1, 2, 4]},
    {"dataset": "amazon-book", "layers": [1, 2, 4]}
]

print(f"🌙 Starting Overnight Execution at {time.strftime('%H:%M:%S')}")
print(f"Directory: {os.getcwd()}")

# 2. Execution Loop
for task in tasks:
    dataset = task["dataset"]
    for L in task["layers"]:
        layer_sizes = ",".join(["64"] * L)
        
        print("\n" + "="*60)
        print(f"RUNNING: {dataset.upper()} | LAYERS: {L} | START: {time.strftime('%H:%M:%S')}")
        print("="*60)
        
        # Build the command exactly as you've been running it
        cmd = [
            "python", "main.py",
            "--dataset", dataset,
            "--n_layers", str(L),
            "--layer_sizes", layer_sizes,
            "--epoch", "400",
            "--eval_interval", "10",
            "--batch_size", "4096",
            "--device", "cpu"
        ]
        
        # Execute and pipe output to this cell
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        
        # Stream the output so you can see progress if you wake up at night
        for line in process.stdout:
            print(line, end="")
            
        process.wait()
        
        if process.returncode == 0:
            print(f"\n✅ COMPLETED: {dataset} L={L} at {time.strftime('%H:%M:%S')}")
        else:
            print(f"\n❌ FAILED: {dataset} L={L}. Moving to next task...")

print("\n" + "☀️" * 15)
print("ALL ABLATION RUNS COMPLETE. READY FOR THE REPORT!")
print("☀️" * 15)

🌙 Starting Overnight Execution at 00:24:18
Directory: /Users/joshua/Desktop/REC_SYS_PROJECT

RUNNING: GOWALLA | LAYERS: 2 | START: 00:24:18
/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          gowalla
  Device:           cpu
  Embedding size:   64
  Layers:           2 x [64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_gowalla_L2

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      29,858
  #Items:      40,981
  #Train:     810,128
  #Test:      217,242
  Density:    0.00066

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(70839, 70839), nnz=1,620,256

[Step 3/5]

In [ ]:
import subprocess
import os
import time

project_dir = "/Users/joshua/Desktop/REC_SYS_PROJECT"
os.chdir(project_dir)

# We are resuming from Yelp L=2 onwards
tasks = [
    {"dataset": "yelp2018", "layers": [2, 4]},
    {"dataset": "amazon-book", "layers": [1, 2, 4]}
]

print(f"🔄 Resuming Execution at {time.strftime('%H:%M:%S')}")

for task in tasks:
    dataset = task["dataset"]
    for L in task["layers"]:
        layer_sizes = ",".join(["64"] * L)
        print(f"\n🚀 STARTING: {dataset.upper()} | L={L} | {time.strftime('%H:%M:%S')}")
        
        cmd = [
            "python", "main.py",
            "--dataset", dataset,
            "--n_layers", str(L),
            "--layer_sizes", layer_sizes,
            "--epoch", "400",
            "--eval_interval", "10",
            "--batch_size", "4096",
            "--device", "cpu"
        ]
        
        # Use a simpler execution to avoid the Jupyter output buffer hang
        result = subprocess.run(cmd, capture_output=False) 
        
        if result.returncode == 0:
            print(f"✅ DONE: {dataset} L={L}")
        else:
            print(f"❌ FAILED: {dataset} L={L}")
        
        time.sleep(10) # 10-second breather for the CPU

🔄 Resuming Execution at 23:30:15

🚀 STARTING: YELP2018 | L=2 | 23:30:15


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          yelp2018
  Device:           cpu
  Embedding size:   64
  Layers:           2 x [64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_yelp2018_L2

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      31,668
  #Items:      38,048
  #Train:   1,237,259
  #Test:      324,147
  Density:    0.00103

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(69716, 69716), nnz=2,474,518

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------
    10 |    0.13394    0.13394   0.000000 |  62.6s |     0.0345     0.0281
    20 |    0.10968    0.10968   0.0

/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          yelp2018
  Device:           cpu
  Embedding size:   64
  Layers:           4 x [64, 64, 64, 64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_yelp2018_L4

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      31,668
  #Items:      38,048
  #Train:   1,237,259
  #Test:      324,147
  Density:    0.00103

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(69716, 69716), nnz=2,474,518

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------
    10 |    0.09664    0.09664   0.000000 | 124.6s |     0.0423     0.0344
    20 |    0.08283    0.082

/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           cpu
  Embedding size:   64
  Layers:           1 x [64]
  Learning rate:    0.0001
  Batch size:       4096
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_amazon-book_L1

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------
    10 |    0.28722    0.28722   0.000001 | 131.8s |     0.0105     0.0087
    20 |    0.18941    0.18941  

/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [1]:
import os
import time
import torch

# 1. Setup
project_dir = "/Users/joshua/Desktop/REC_SYS_PROJECT"
os.chdir(project_dir)

# 2. Check for GPU
if torch.backends.mps.is_available():
    device = "mps"
    print("🚀 GPU (MPS) DETECTED. Switching to high-speed training.")
else:
    device = "cpu"
    print("⚠️ MPS not found, falling back to CPU.")

# 3. The Final Two Tasks for Amazon-Book
tasks = [
    {"L": 2, "layer_sizes": "64,64"},
    {"L": 4, "layer_sizes": "64,64,64,64"}
]

for task in tasks:
    print(f"\n" + "="*60)
    print(f"⚡️ GPU RUN: AMAZON-BOOK | L={task['L']} | START: {time.strftime('%H:%M:%S')}")
    print("="*60)
    
    # Using --device mps and a slightly smaller batch_size for GPU stability
    cmd = (
        f"python main.py --dataset amazon-book --n_layers {task['L']} "
        f"--layer_sizes {task['layer_sizes']} --epoch 400 --eval_interval 20 "
        f"--batch_size 2048 --device {device}"
    )
    
    os.system(cmd)
    print(f"✅ FINISHED L={task['L']}")

print("\n🎉 ALL REPLICATION DATA SECURED FOR TOMORROW!")

🚀 GPU (MPS) DETECTED. Switching to high-speed training.

⚡️ GPU RUN: AMAZON-BOOK | L=2 | START: 13:54:05


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           mps
  Embedding size:   64
  Layers:           2 x [64, 64]
  Learning rate:    0.0001
  Batch size:       2048
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_amazon-book_L2

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------


Traceback (most recent call last):
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 174, in <module>
    main()
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 126, in main
    loss, bv, rv = bpr_loss(u_emb, p_emb, n_emb, u_ego, p_ego, n_ego, args.reg)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 39, in bpr_loss
    return bpr + reg, bpr.item(), reg.item()
                      ^^^^^^^^^^
KeyboardInterrupt


✅ FINISHED L=2

⚡️ GPU RUN: AMAZON-BOOK | L=4 | START: 14:17:03


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           mps
  Embedding size:   64
  Layers:           4 x [64, 64, 64, 64]
  Learning rate:    0.0001
  Batch size:       2048
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_amazon-book_L4

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------


Traceback (most recent call last):
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 174, in <module>
    main()
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 126, in main
    loss, bv, rv = bpr_loss(u_emb, p_emb, n_emb, u_ego, p_ego, n_ego, args.reg)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 39, in bpr_loss
    return bpr + reg, bpr.item(), reg.item()
                      ^^^^^^^^^^
KeyboardInterrupt


✅ FINISHED L=4

🎉 ALL REPLICATION DATA SECURED FOR TOMORROW!


In [ ]:
import os
import time
import torch

# 1. Setup Directory
project_dir = "/Users/joshua/Desktop/REC_SYS_PROJECT"
os.chdir(project_dir)

# 2. Force GPU Detection (MPS for Mac M4 Pro)
if torch.backends.mps.is_available():
    device = "mps"
    print("🚀 GPU (MPS) DETECTED. Switching to high-speed training.")
else:
    device = "cpu"
    print("⚠️ MPS not found, falling back to CPU.")

# 3. Define the Missing Tasks (Resuming Amazon-Book)
# We are running L=2 and L=4 to complete Table 3
tasks = [
    {"L": 2, "layers": "64,64"},
    {"L": 4, "layers": "64,64,64,64"}
]

for task in tasks:
    L = task["L"]
    layer_sizes = task["layers"]
    
    print("\n" + "="*60)
    print(f"⚡️ GPU RUN: AMAZON-BOOK | L={L} | START: {time.strftime('%H:%M:%S')}")
    print("="*60)
    
    # Using --device mps
    # eval_interval 20 saves a lot of time on large datasets
    # batch_size 2048 is more stable for GPU memory
    cmd = (
        f"python main.py --dataset amazon-book --n_layers {L} "
        f"--layer_sizes {layer_sizes} --epoch 400 --eval_interval 20 "
        f"--batch_size 2048 --device {device}"
    )
    
    # Execute and stream output
    os.system(cmd)
    print(f"✅ COMPLETED: Amazon-Book L={L}")

print("\n" + "🎉" * 10)
print("ALL DATA COLLECTED! YOUR REPLICATION IS READY FOR THE SHOWCASE!")

🚀 GPU (MPS) DETECTED. Switching to high-speed training.

⚡️ GPU RUN: AMAZON-BOOK | L=2 | START: 14:17:22


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           mps
  Embedding size:   64
  Layers:           2 x [64, 64]
  Learning rate:    0.0001
  Batch size:       2048
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_amazon-book_L2

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------
    20 |    0.11957    0.11956   0.000001 | 202.6s |     0.0165     0.0129


Traceback (most recent call last):
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 174, in <module>
    main()
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 126, in main
    loss, bv, rv = bpr_loss(u_emb, p_emb, n_emb, u_ego, p_ego, n_ego, args.reg)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joshua/Desktop/REC_SYS_PROJECT/main.py", line 39, in bpr_loss
    return bpr + reg, bpr.item(), reg.item()
                      ^^^^^^^^^^
KeyboardInterrupt


✅ COMPLETED: Amazon-Book L=2

⚡️ GPU RUN: AMAZON-BOOK | L=4 | START: 19:11:51


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import os
import time
import torch

# 1. Setup Directory
project_dir = "/Users/joshua/Desktop/REC_SYS_PROJECT"
os.chdir(project_dir)

# 2. Force GPU Detection (MPS for Mac M4 Pro)
if torch.backends.mps.is_available():
    device = "mps"
    print("🚀 GPU (MPS) DETECTED. Switching to high-speed training.")
else:
    device = "cpu"
    print("⚠️ MPS not found, falling back to CPU.")

# 3. Define the Missing Tasks (Resuming Amazon-Book)
# We are running L=2 and L=4 to complete Table 3
tasks = [
    {"L": 2, "layers": "64,64"},
    {"L": 4, "layers": "64,64,64,64"}
]

for task in tasks:
    L = task["L"]
    layer_sizes = task["layers"]
    
    print("\n" + "="*60)
    print(f"⚡️ GPU RUN: AMAZON-BOOK | L={L} | START: {time.strftime('%H:%M:%S')}")
    print("="*60)
    
    # Using --device mps
    # eval_interval 20 saves a lot of time on large datasets
    # batch_size 2048 is more stable for GPU memory
    cmd = (
        f"python main.py --dataset amazon-book --n_layers {L} "
        f"--layer_sizes {layer_sizes} --epoch 400 --eval_interval 20 "
        f"--batch_size 2048 --device {device}"
    )
    
    # Execute and stream output
    os.system(cmd)
    print(f"✅ COMPLETED: Amazon-Book L={L}")

print("\n" + "🎉" * 10)
print("ALL DATA COLLECTED! YOUR REPLICATION IS READY FOR THE SHOWCASE!")

🚀 GPU (MPS) DETECTED. Switching to high-speed training.

⚡️ GPU RUN: AMAZON-BOOK | L=2 | START: 19:12:10


/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  NGCF — Neural Graph Collaborative Filtering
  Faithful PyTorch Reproduction (Wang et al., SIGIR 2019)
  Dataset:          amazon-book
  Device:           mps
  Embedding size:   64
  Layers:           2 x [64, 64]
  Learning rate:    0.0001
  Batch size:       2048
  L2 reg (lambda):  1e-05
  Run Tag:          ngcf_amazon-book_L2

[Step 1/5] Loading dataset ...
  Dataset loaded
  #Users:      52,643
  #Items:      91,599
  #Train:   2,380,730
  #Test:      603,378
  Density:    0.00049

[Step 2/5] Building normalized Laplacian (Eq. 8) ...
  Laplacian: shape=(144242, 144242), nnz=4,761,460

[Step 3/5] Initializing NGCF model ...

[Step 4/5] Training (max 400 epochs) ...
----------------------------------------------------------------------
 Epoch |       Loss        BPR        Reg |   Time |  recall@20    ndcg@20
----------------------------------------------------------------------
    20 |    0.11958    0.11958   0.000001 | 126.1s |     0.0165     0.0129
    40 |    0.08239    0.082

/Users/joshua/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
